In [1]:
import numpy as np #수치 계산을 위한 라이브러리
import addcopyfighandler #복사 붙여 넣기 등의 기능 가능
import matplotlib.pyplot as plt # 그래프 그리기 위한 라이브러리
import mne #뇌파 데이터 분석을 위한 라이브러리

In [2]:
sample_data_folder = mne.datasets.sample.data_path() # 폴더 경로 설정
sample_data_raw_file = sample_data_folder / "MEG" / "sample" / "sample_audvis_raw.fif"
# raw 데이터 파일 경로 설정
raw = mne.io.read_raw_fif(sample_data_raw_file) # raw 데이터 설정
raw.crop(tmax=60).load_data() # raw 데이터 자르고(61초까지) 로드

Opening raw data file C:\Users\user\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 36037  =      0.000 ...    60.000 secs...


<Raw | sample_audvis_raw.fif, 376 x 36038 (60.0 s), ~106.6 MiB, data loaded>

In [3]:
raw.copy().pick(picks="stim").plot(start = 3, duration = 6) 
# raw 데이터 복사해서 stim(자극) 채널만 선택해서 3초부터 6초 동안(9초까지)의 데이터를 그래프로 그리기

Using qt as 2D backend.


Channels marked as bad:
none
Channels marked as bad:
['MEG 2443', 'EEG 053', np.str_('MEG 0133')]
Channels marked as bad:
['MEG 2443', 'EEG 053', np.str_('MEG 0133')]


In [4]:
events = mne.find_events(raw, stim_channel = "STI 014")
#raw 데이터에서 stim channel 중 summation channel "STI 014"에서 이벤트 찾기
print(events[:5])
# 찾은 이벤트 중 처음 5개 출력

Finding events on: STI 014
86 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]
[[27977     0     2]
 [28345     0     3]
 [28771     0     1]
 [29219     0     4]
 [29652     0     2]]


In [5]:
testing_data_folder = mne.datasets.testing.data_path() # 폴더 경로 설정
eeglab_raw_file = testing_data_folder / "EEGLAB" / "test_raw.set"
# raw 데이터 파일 경로 설정
eeglab_raw = mne.io.read_raw_eeglab(eeglab_raw_file) # raw 데이터 설정
print(eeglab_raw.annotations) # raw 데이터의 annotations(주석) 출력

Reading C:\Users\user\mne_data\MNE-testing-data\EEGLAB\test_raw.fdt
<Annotations | 154 segments: rt (74), square (80)>


In [6]:
print(len(eeglab_raw.annotations))
# raw 데이터의 annotations(주석) 개수 출력
print(set(eeglab_raw.annotations.duration))
# raw 데이터의 annotations(주석) 지속 시간 출력
print(set(eeglab_raw.annotations.description))
# raw 데이터의 annotations(주석) 설명 출력
print(eeglab_raw.annotations.onset[0])
# raw 데이터의 첫 번째 annotations(주석) 시작 시간 출력

154
{np.float64(0.0)}
{np.str_('rt'), np.str_('square')}
1.000068


In [7]:
events_from_annot, event_dict = mne.events_from_annotations(eeglab_raw)
# raw 데이터의 annotations(주석)에서 이벤트 추출하기
print(event_dict)
# 추출한 event의 array 출력 array의 설명과 정수 ID 매핑
print(events_from_annot[:5])
# 추출한 이벤트 중 처음 5개 출력
# onset은 1열 duration은 2열 description은 3열

Used Annotations descriptions: [np.str_('rt'), np.str_('square')]
{np.str_('rt'): 1, np.str_('square'): 2}
[[128   0   2]
 [217   0   2]
 [267   0   1]
 [602   0   2]
 [659   0   1]]


In [8]:
custom_mapping = {"rt": 77, "square": 42}
# custom mapping 설정, "rt"는 77로, "square"는 42로 매핑
(events_from_annot, event_dict) = mne.events_from_annotations(
    eeglab_raw, event_id = custom_mapping
)
# raw 데이터의 annotations(주석)에서 이벤트 추출하기, custom mapping 적용
print(event_dict)
# 추출한 event의 array 출력 array의 설명과 정수 ID 매핑
print(events_from_annot[:5])
# 추출한 이벤트 중 처음 5개 출력

Used Annotations descriptions: [np.str_('rt'), np.str_('square')]
{np.str_('rt'): 77, np.str_('square'): 42}
[[128   0  42]
 [217   0  42]
 [267   0  77]
 [602   0  42]
 [659   0  77]]


In [9]:
mapping = {
    1: "ayditory/left", # 정수 ID 1을 "ayditory/left"로 매핑
    2: "ayditory/right", # 정수 ID 2를 "ayditory/right"로 매핑
    3: "visual/left", # 정수 ID 3을 "visual/left"로 매핑
    4: "visual/right", # 정수 ID 4을 "visual/right"로 매핑
    5: "smiley", # 정수 ID 5을 "smiley"로 매핑
    32: "buttonpress", # 정수 ID 32을 "buttonpress"로 매핑
} 
# custom mapping 설정, 정수 ID를 설명으로 매핑
annot_from_events = mne.annotations_from_events( # 이벤트에서 annotations(주석) 생성하기
    events=events, # 이벤트 배열
    event_desc=mapping, # 이벤트 설명 매핑
    sfreq=raw.info["sfreq"], # 샘플링 주파수
    orig_time=raw.info["meas_date"], # 원래 시간 정보
)
raw.set_annotations(annot_from_events)
# raw 데이터에 생성한 annotations(주석) 설정하기

<Raw | sample_audvis_raw.fif, 376 x 36038 (60.0 s), ~106.6 MiB, data loaded>

In [12]:
raw.plot(start=5, duration =5)
# raw 데이터 그래프로 그리기, 5초부터 5초 동안(10초까지) 보여주기

In [11]:
#create the Rem annotations
rem_annot = mne.Annotations(onset=[5, 41], duration=[16, 11], description =["REM"] * 2)
# Rem annotations 생성하기, 시작 시간은 5초와 41초, 지속 시간은 16초와 11초, 설명은 "REM"으로 설정
# ["REM"] *2 ["REM", "REM"]와 같은 의미 onset과 duration이 2개씩 있으므로 description도 2개 필요
raw.set_annotations(rem_annot) # raw 데이터에 생성한 annotations(주석) 설정하기
(rem_events, rem_event_dict) = mne.events_from_annotations(raw, chunk_duration = 1.5)
# raw 데이터의 annotations(주석)에서 이벤트 추출하기, chunk_duration을 1.5초로 설정

Used Annotations descriptions: [np.str_('REM')]


In [ ]:
print(np.round((rem_events[:, 0] - raw.first_samp) / raw.info["sfreq"], 3))
# rem_events[:, 0]: 이벤트 배열에서 모든 이벤트의 샘플 번호(첫 번째 열)만 추출하기
# raw.first_samp: raw 데이터의 첫 번째 샘플 번호 
# -> 이벤트의 샘플 번호에서 raw 데이터의 첫 번째 샘플 번호를 빼면 이벤트가 raw 데이터의 시작부터 얼마나 떨어져 있는지 알 수 있음
# raw.info["sfreq"]: raw 데이터의 샘플링 주파수 
# -> 이벤트가 raw 데이터의 시작부터 얼마나 떨어져 있는지 초 단위로 계산하기 위해 샘플 번호를 샘플링 주파수로 나누기
# np.round(..., 3): 계산된 이벤트의 시간 정보를 소수점 3자리까지 반올림하기

[ 5.     6.5    8.     9.5   11.    12.501 14.001 15.501 16.999 18.499
 41.    42.5   44.    45.5   47.    48.5   50.   ]
